In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
================================================================================
📥 АВТОМАТИЧЕСКОЕ ОБНОВЛЕНИЕ КУРСОВ ВАЛЮТ ЦБ РФ В POSTGRESQL
================================================================================

ИСТОЧНИК: Центральный банк Российской Федерации (www.cbr.ru)
ПРИЕМНИК: Таблица public.exchange_rate в PostgreSQL
СЕРВЕР: 109.196.102.91
БАЗА ДАННЫХ: default_db

ОПИСАНИЕ ТАБЛИЦЫ exchange_rate:
--------------------------------------------------------------------------------
Поле             | Тип данных | Описание
--------------------------------------------------------------------------------
date             | DATE       | Дата курса
currency         | VARCHAR(3) | Код валюты (USD, EUR, и т.д.)
nominal          | INTEGER    | Номинал (количество единиц валюты)
rate             | NUMERIC(10,4) | Курс за 1 единицу валюты
--------------------------------------------------------------------------------
Составной PRIMARY KEY: (date, currency)
"""

import requests
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import time
import psycopg2
import logging
from tabulate import tabulate
import warnings
warnings.filterwarnings("ignore", category=requests.packages.urllib3.exceptions.InsecureRequestWarning)

# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Конфигурация
class Config:
    DB_HOST = "109.196.102.91"
    DB_NAME = "default_db"
    DB_USER = "maksv"
    DB_PASSWORD = r"p27Oqwp4Y,X^O^"
    DB_SCHEMA = "public"
    DB_TABLE = "exchange_rate"  # Изменено на exchange_rate
    
    # ID валют в системе ЦБ
    CURRENCY_IDS = {
        'USD': 'R01235',
        'EUR': 'R01239',
        'GBP': 'R01035',
        'JPY': 'R01820',
        'CNY': 'R01375',
        'CHF': 'R01775',
        'KZT': 'R01335',
        'BYN': 'R01090'
    }
    
    # Список валют для парсинга
    CURRENCIES = ['USD', 'EUR', 'GBP', 'JPY', 'CNY', 'CHF', 'KZT', 'BYN']


def get_db_connection():
    """
    Создание подключения к базе данных
    """
    try:
        conn = psycopg2.connect(
            host=Config.DB_HOST,
            database=Config.DB_NAME,
            user=Config.DB_USER,
            password=Config.DB_PASSWORD
        )
        return conn
    except Exception as e:
        logger.error(f"❌ Ошибка подключения к БД: {e}")
        raise


def check_and_create_table():
    """
    Проверка существования таблицы exchange_rate и создание при необходимости
    """
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        # Проверяем существование таблицы
        cur.execute("""
            SELECT EXISTS (
                SELECT FROM information_schema.tables 
                WHERE table_schema = %s AND table_name = %s
            )
        """, (Config.DB_SCHEMA, Config.DB_TABLE))
        
        exists = cur.fetchone()[0]
        
        if not exists:
            logger.info(f"📋 Таблица {Config.DB_TABLE} не существует. Создаем...")
            
            # Создаем таблицу с составным PRIMARY KEY
            cur.execute(f"""
                CREATE TABLE {Config.DB_SCHEMA}.{Config.DB_TABLE} (
                    date DATE NOT NULL,
                    currency VARCHAR(3) NOT NULL,
                    nominal INTEGER NOT NULL,
                    rate NUMERIC(10,4) NOT NULL,
                    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    PRIMARY KEY (date, currency)
                )
            """)
            
            conn.commit()
            logger.info(f"✅ Таблица {Config.DB_TABLE} успешно создана с PRIMARY KEY (date, currency)")
            return True
        else:
            # Проверяем структуру таблицы и добавляем PRIMARY KEY если нужно
            cur.execute("""
                SELECT EXISTS (
                    SELECT 1
                    FROM information_schema.table_constraints tc
                    WHERE tc.constraint_type = 'PRIMARY KEY'
                        AND tc.table_schema = %s
                        AND tc.table_name = %s
                )
            """, (Config.DB_SCHEMA, Config.DB_TABLE))
            
            has_primary_key = cur.fetchone()[0]
            
            if not has_primary_key:
                logger.warning(f"⚠️ Таблица {Config.DB_TABLE} существует, но нет PRIMARY KEY")
                logger.info(f"🔄 Добавляем PRIMARY KEY (date, currency)...")
                
                # Добавляем PRIMARY KEY
                cur.execute(f"""
                    ALTER TABLE {Config.DB_SCHEMA}.{Config.DB_TABLE}
                    ADD CONSTRAINT exchange_rate_pkey PRIMARY KEY (date, currency)
                """)
                
                conn.commit()
                logger.info(f"✅ PRIMARY KEY успешно добавлен в таблицу {Config.DB_TABLE}")
            
            # Проверяем наличие колонки updated_at
            cur.execute("""
                SELECT EXISTS (
                    SELECT FROM information_schema.columns 
                    WHERE table_schema = %s 
                        AND table_name = %s 
                        AND column_name = 'updated_at'
                )
            """, (Config.DB_SCHEMA, Config.DB_TABLE))
            
            has_updated_at = cur.fetchone()[0]
            
            if not has_updated_at:
                logger.info(f"🔄 Добавляем колонку updated_at в таблицу {Config.DB_TABLE}...")
                cur.execute(f"""
                    ALTER TABLE {Config.DB_SCHEMA}.{Config.DB_TABLE}
                    ADD COLUMN updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                """)
                conn.commit()
                logger.info(f"✅ Колонка updated_at успешно добавлена в таблицу {Config.DB_TABLE}")
            
            logger.info(f"✅ Таблица {Config.DB_TABLE} существует и готова к работе")
            return True
        
    except Exception as e:
        logger.error(f"❌ Ошибка при проверке/создании таблицы {Config.DB_TABLE}: {e}")
        if conn:
            conn.rollback()
        return False
    finally:
        if conn:
            conn.close()


def get_existing_dates_from_db(currency=None):
    """
    Получение существующих дат из таблицы exchange_rate для одной или всех валют
    """
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        if currency:
            cur.execute(f"""
                SELECT date 
                FROM {Config.DB_SCHEMA}.{Config.DB_TABLE}
                WHERE currency = %s
            """, (currency,))
            logger.info(f"📋 Получаем существующие даты для {currency} из {Config.DB_TABLE}")
        else:
            cur.execute(f"""
                SELECT date, currency 
                FROM {Config.DB_SCHEMA}.{Config.DB_TABLE}
            """)
            result = cur.fetchall()
            # Возвращаем словарь: ключ - валюта, значение - множество дат
            existing_by_currency = {}
            for date, curr in result:
                if curr not in existing_by_currency:
                    existing_by_currency[curr] = set()
                existing_by_currency[curr].add(date)
            logger.info(f"📋 В таблице {Config.DB_TABLE} найдены записи для {len(existing_by_currency)} валют")
            return existing_by_currency
        
        existing_dates = {row[0] for row in cur.fetchall()}
        logger.info(f"📋 Для {currency} в таблице {Config.DB_TABLE} найдено {len(existing_dates)} существующих записей")
        
        return existing_dates
        
    except Exception as e:
        logger.error(f"❌ Ошибка при получении существующих дат из таблицы {Config.DB_TABLE}: {e}")
        return set() if currency else {}
    finally:
        if conn:
            conn.close()


def get_db_statistics():
    """
    Получение статистики из таблицы exchange_rate
    """
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        cur.execute(f"""
            SELECT 
                COUNT(*) as total_records,
                COUNT(DISTINCT currency) as currencies_count,
                MIN(date) as first_date,
                MAX(date) as last_date
            FROM {Config.DB_SCHEMA}.{Config.DB_TABLE}
        """)
        
        general_stats = cur.fetchone()
        
        # Статистика по каждой валюте
        cur.execute(f"""
            SELECT 
                currency,
                COUNT(*) as records,
                MIN(date) as first_date,
                MAX(date) as last_date,
                MIN(rate) as min_rate,
                MAX(rate) as max_rate,
                ROUND(AVG(rate)::numeric, 4) as avg_rate
            FROM {Config.DB_SCHEMA}.{Config.DB_TABLE}
            GROUP BY currency
            ORDER BY currency
        """)
        
        currency_stats = cur.fetchall()
        
        return {
            'total_records': general_stats[0] or 0,
            'currencies_count': general_stats[1] or 0,
            'first_date': general_stats[2],
            'last_date': general_stats[3],
            'currency_stats': currency_stats
        }
        
    except Exception as e:
        logger.error(f"❌ Ошибка при получении статистики из таблицы {Config.DB_TABLE}: {e}")
        return None
    finally:
        if conn:
            conn.close()


def display_statistics(stats, title="📊 СТАТИСТИКА КУРСОВ ВАЛЮТ В ТАБЛИЦЕ exchange_rate"):
    """
    Отображение статистики в табличном виде
    """
    if not stats or stats['total_records'] == 0:
        print("\n❌ В таблице exchange_rate нет данных по курсам валют")
        return
    
    print("\n" + "="*90)
    print(title)
    print("="*90)
    
    general_data = [
        ["Всего записей", f"{stats['total_records']}"],
        ["Количество валют", f"{stats['currencies_count']}"],
        ["Период данных", f"{stats['first_date']} - {stats['last_date']}"]
    ]
    
    print(tabulate(general_data, headers=["Показатель", "Значение"], tablefmt="grid"))
    
    if stats['currency_stats']:
        print("\n📈 СТАТИСТИКА ПО ВАЛЮТАМ:")
        currency_table = []
        for curr in stats['currency_stats']:
            currency_table.append([
                curr[0], curr[1], curr[2], curr[3], f"{curr[4]:.4f}", f"{curr[5]:.4f}", f"{curr[6]:.4f}"
            ])
        
        print(tabulate(currency_table, 
                      headers=["Валюта", "Записей", "С", "По", "Мин", "Макс", "Средн."], 
                      tablefmt="grid"))
    
    print("="*90)


def parse_currency_cbr(currency_code, start_date, end_date):
    """
    Парсинг курса валюты с сайта ЦБ РФ
    
    Параметры:
    currency_code: код валюты (USD, EUR, GBP, JPY, CNY и т.д.)
    start_date: начальная дата (datetime)
    end_date: конечная дата (datetime)
    """
    
    currency_id = Config.CURRENCY_IDS.get(currency_code, 'R01235')
    
    url = "http://www.cbr.ru/scripts/XML_dynamic.asp"
    date_format = "%d/%m/%Y"
    
    params = {
        'date_req1': start_date.strftime(date_format),
        'date_req2': end_date.strftime(date_format),
        'VAL_NM_RQ': currency_id
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.encoding = 'windows-1251'
        root = ET.fromstring(response.text)
        
        data = []
        for record in root.findall('Record'):
            date = datetime.strptime(record.get('Date'), "%d.%m.%Y")
            value = float(record.find('Value').text.replace(',', '.'))
            nominal = int(record.find('Nominal').text)
            
            # Расчет курса за 1 единицу валюты
            rate = value / nominal
            
            data.append({
                'date': date,
                'nominal': nominal,
                'rate': rate,
                'currency': currency_code
            })
        
        if data:
            df = pd.DataFrame(data)
            logger.info(f"✅ {currency_code}: получено {len(df)} записей за период {start_date.date()} - {end_date.date()}")
            return df
        else:
            logger.warning(f"⚠️ {currency_code}: нет данных за указанный период")
            return pd.DataFrame()
            
    except Exception as e:
        logger.error(f"❌ Ошибка при парсинге {currency_code}: {e}")
        return pd.DataFrame()


def get_new_records_only(df, existing_dates):
    """
    Фильтрация только новых записей, которых нет в базе
    """
    if df.empty:
        return df
    
    if not existing_dates:
        logger.info(f"📋 {df['currency'].iloc[0]}: таблица exchange_rate пуста - будут добавлены все записи")
        return df
    
    # Фильтруем записи, которых нет в existing_dates
    new_records = df[~df['date'].dt.date.isin(existing_dates)]
    
    currency = df['currency'].iloc[0]
    if len(new_records) > 0:
        logger.info(f"🆕 {currency}: найдено {len(new_records)} новых записей для добавления в exchange_rate")
        if len(new_records) < len(df):
            latest_existing = max(existing_dates) if existing_dates else None
            logger.info(f"📅 {currency}: последняя существующая дата в exchange_rate: {latest_existing}")
            logger.info(f"📅 {currency}: новые данные до: {new_records['date'].max().date()}")
    else:
        latest_existing = max(existing_dates) if existing_dates else None
        logger.info(f"✅ {currency}: данные в exchange_rate актуальны. Последняя запись: {latest_existing}")
    
    return new_records


def save_to_postgresql(df):
    """
    Сохранение только новых данных в таблицу exchange_rate
    """
    if df is None or df.empty:
        return 0
    
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        records_saved = 0
        currency = df['currency'].iloc[0]
        
        for _, row in df.iterrows():
            date_obj = row['date'].date()
            currency_code = row['currency']
            nominal = int(row['nominal'])
            rate_value = round(float(row['rate']), 4)
            
            # Вставляем новую запись
            insert_sql = f"""
                INSERT INTO {Config.DB_SCHEMA}.{Config.DB_TABLE} 
                (date, currency, nominal, rate, updated_at)
                VALUES (%s, %s, %s, %s, CURRENT_TIMESTAMP)
                ON CONFLICT (date, currency) DO NOTHING
            """
            
            cur.execute(insert_sql, (date_obj, currency_code, nominal, rate_value))
            if cur.rowcount > 0:
                records_saved += 1
        
        conn.commit()
        
        if records_saved > 0:
            logger.info(f"✅ {currency}: добавлено {records_saved} новых записей в таблицу exchange_rate")
        
        return records_saved
        
    except Exception as e:
        logger.error(f"❌ Ошибка при сохранении {currency} в таблицу exchange_rate: {e}")
        if conn:
            conn.rollback()
        return 0
    finally:
        if conn:
            conn.close()


def update_exchange_rates():
    """
    Основная функция обновления курсов валют в таблице exchange_rate
    Добавляет только новые записи, которых нет в базе
    """
    print("\n" + "="*100)
    print("💰 АВТОМАТИЧЕСКОЕ ОБНОВЛЕНИЕ КУРСОВ ВАЛЮТ ЦБ РФ")
    print("="*100)
    print(f"🖥️  Сервер: {Config.DB_HOST}")
    print(f"🗄️  База данных: {Config.DB_NAME}")
    print(f"📋 Таблица: {Config.DB_SCHEMA}.{Config.DB_TABLE}")
    print("="*100)
    
    # Проверяем/создаем таблицу
    if not check_and_create_table():
        logger.error("❌ Не удалось настроить таблицу exchange_rate")
        return None
    
    # Получаем текущую статистику
    current_stats = get_db_statistics()
    display_statistics(current_stats, "📊 ТЕКУЩАЯ СТАТИСТИКА В ТАБЛИЦЕ exchange_rate")
    
    # Получаем существующие записи по валютам
    existing_by_currency = get_existing_dates_from_db()
    
    # Период для загрузки (с 1992 года)
    start_date = datetime(1992, 7, 1)
    end_date = datetime.now()
    
    logger.info(f"\n🔄 Период загрузки: {start_date.date()} - {end_date.date()}")
    
    total_new_records = 0
    results = {}
    
    # Парсинг каждой валюты
    for i, currency in enumerate(Config.CURRENCIES, 1):
        print(f"\n{'='*60}")
        logger.info(f"🔄 Обработка {currency}... ({i}/{len(Config.CURRENCIES)})")
        
        # Получаем существующие даты для этой валюты
        existing_dates = existing_by_currency.get(currency, set()) if isinstance(existing_by_currency, dict) else set()
        
        # Загружаем данные с сайта ЦБ
        df_all = parse_currency_cbr(currency, start_date, end_date)
        
        if df_all.empty:
            logger.warning(f"⚠️ {currency}: нет данных")
            results[currency] = 0
            continue
        
        # Фильтруем только новые записи
        df_new = get_new_records_only(df_all, existing_dates)
        
        if df_new.empty:
            results[currency] = 0
            continue
        
        # Сохраняем новые записи
        records_saved = save_to_postgresql(df_new)
        total_new_records += records_saved
        results[currency] = records_saved
        
        # Пауза между запросами
        if i < len(Config.CURRENCIES):
            time.sleep(1)
    
    # Получаем обновленную статистику
    new_stats = get_db_statistics()
    display_statistics(new_stats, "📊 ОБНОВЛЕННАЯ СТАТИСТИКА В ТАБЛИЦЕ exchange_rate")
    
    print("\n" + "="*80)
    print("✅ ОПЕРАЦИЯ ЗАВЕРШЕНА")
    print("="*80)
    print(f"📥 Всего добавлено в таблицу exchange_rate: {total_new_records} записей")
    
    if total_new_records > 0:
        print("\n📊 ПО ВАЛЮТАМ:")
        for currency, saved in results.items():
            if saved > 0:
                print(f"  {currency}: +{saved} записей")
    
    print(f"\n📊 Всего в таблице exchange_rate: {new_stats['total_records']} записей")
    print(f"📅 Период: {new_stats['first_date']} - {new_stats['last_date']}")
    print("="*80)
    
    return {
        'total_new_records': total_new_records,
        'results': results,
        'total_in_db': new_stats['total_records'],
        'first_date': new_stats['first_date'],
        'last_date': new_stats['last_date']
    }


def main():
    """
    Основная функция - автоматическое обновление курсов валют в таблице exchange_rate
    """
    # Запускаем автоматическое обновление
    results = update_exchange_rates()


if __name__ == "__main__":
    main()


💰 АВТОМАТИЧЕСКОЕ ОБНОВЛЕНИЕ КУРСОВ ВАЛЮТ ЦБ РФ
🖥️  Сервер: 109.196.102.91
🗄️  База данных: default_db
📋 Таблица: public.exchange_rate


2026-03-01 16:47:27,079 - __main__ - WARNING - ⚠️ Таблица exchange_rate существует, но нет PRIMARY KEY
2026-03-01 16:47:27,085 - __main__ - INFO - 🔄 Добавляем PRIMARY KEY (date, currency)...
2026-03-01 16:47:27,233 - __main__ - INFO - ✅ PRIMARY KEY успешно добавлен в таблицу exchange_rate
2026-03-01 16:47:27,333 - __main__ - INFO - 🔄 Добавляем колонку updated_at в таблицу exchange_rate...
2026-03-01 16:47:27,350 - __main__ - INFO - ✅ Колонка updated_at успешно добавлена в таблицу exchange_rate
2026-03-01 16:47:27,350 - __main__ - INFO - ✅ Таблица exchange_rate существует и готова к работе



📊 ТЕКУЩАЯ СТАТИСТИКА В ТАБЛИЦЕ exchange_rate
+------------------+-------------------------+
| Показатель       | Значение                |
+==================+=========================+
| Всего записей    | 57903                   |
+------------------+-------------------------+
| Количество валют | 8                       |
+------------------+-------------------------+
| Период данных    | 1992-07-01 - 2026-02-25 |
+------------------+-------------------------+

📈 СТАТИСТИКА ПО ВАЛЮТАМ:
+----------+-----------+------------+------------+---------+------------+----------+
| Валюта   |   Записей | С          | По         |     Мин |       Макс |   Средн. |
+==========+===========+============+============+=========+============+==========+
| BYN      |      7349 | 1994-01-14 | 2026-02-25 |  0      |    38.8656 |   9.4648 |
+----------+-----------+------------+------------+---------+------------+----------+
| CHF      |      7779 | 1992-07-01 | 2026-02-25 |  3.987  |  4519.82   | 354.79

2026-03-01 16:47:29,638 - __main__ - INFO - 📋 В таблице exchange_rate найдены записи для 8 валют
2026-03-01 16:47:29,649 - __main__ - INFO - 
🔄 Период загрузки: 1992-07-01 - 2026-03-01
2026-03-01 16:47:29,656 - __main__ - INFO - 🔄 Обработка USD... (1/8)


2026-03-01 16:47:30,022 - __main__ - INFO - ✅ USD: получено 7782 записей за период 1992-07-01 - 2026-03-01
2026-03-01 16:47:30,028 - __main__ - INFO - 📋 USD: таблица exchange_rate пуста - будут добавлены все записи
2026-03-01 16:48:39,887 - __main__ - INFO - ✅ USD: добавлено 3 новых записей в таблицу exchange_rate
2026-03-01 16:48:40,900 - __main__ - INFO - 🔄 Обработка EUR... (2/8)


...

2026-03-01 16:52:10,609 - __main__ - INFO - ✅ CNY: получено 5076 записей за период 1992-07-01 - 2026-03-01
2026-03-01 16:52:10,614 - __main__ - INFO - 📋 CNY: таблица exchange_rate пуста - будут добавлены все записи
2026-03-01 16:52:54,540 - __main__ - INFO - ✅ CNY: добавлено 3 новых записей в таблицу exchange_rate
2026-03-01 16:52:55,556 - __main__ - INFO - 🔄 Обработка CHF... (6/8)


2026-03-01 16:52:55,897 - __main__ - INFO - ✅ CHF: получено 7782 записей за период 1992-07-01 - 2026-03-01
2026-03-01 16:52:55,903 - __main__ - INFO - 📋 CHF: таблица exchange_rate пуста - будут добавлены все записи
2026-03-01 16:54:03,306 - __main__ - INFO - ✅ CHF: добавлено 3 новых записей в таблицу exchange_rate
2026-03-01 16:54:04,309 - __main__ - INFO - 🔄 Обработка KZT... (7/8)


2026-03-01 16:54:04,679 - __main__ - INFO - ✅ KZT: получено 7639 записей за период 1992-07-01 - 2026-03-01
2026-03-01 16:54:04,687 - __main__ - INFO - 📋 KZT: таблица exchange_rate пуста - будут добавлены все записи
2026-03-01 16:55:11,199 - __main__ - INFO - ✅ KZT: добавлено 3 новых записей в таблицу exchange_rate
2026-03-01 16:55:12,202 - __main__ - INFO - 🔄 Обработка BYN... (8/8)


2026-03-01 16:55:12,596 - __main__ - INFO - ✅ BYN: получено 7352 записей за период 1992-07-01 - 2026-03-01
2026-03-01 16:55:12,597 - __main__ - INFO - 📋 BYN: таблица exchange_rate пуста - будут добавлены все записи
2026-03-01 16:56:17,123 - __main__ - INFO - ✅ BYN: добавлено 3 новых записей в таблицу exchange_rate



📊 ОБНОВЛЕННАЯ СТАТИСТИКА В ТАБЛИЦЕ exchange_rate
+------------------+-------------------------+
| Показатель       | Значение                |
+==================+=========================+
| Всего записей    | 57927                   |
+------------------+-------------------------+
| Количество валют | 8                       |
+------------------+-------------------------+
| Период данных    | 1992-07-01 - 2026-02-28 |
+------------------+-------------------------+

📈 СТАТИСТИКА ПО ВАЛЮТАМ:
+----------+-----------+------------+------------+---------+------------+----------+
| Валюта   |   Записей | С          | По         |     Мин |       Макс |   Средн. |
+==========+===========+============+============+=========+============+==========+
| BYN      |      7352 | 1994-01-14 | 2026-02-28 |  0      |    38.8656 |   9.472  |
+----------+-----------+------------+------------+---------+------------+----------+
| CHF      |      7782 | 1992-07-01 | 2026-02-28 |  3.987  |  4519.82   | 35